In [0]:
from pyspark.sql import functions as F

df = spark.read.table("databricks_catalog.silver_schema.customers")

df.printSchema()
display(df)
print("Rows:", df.count())

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- email_domain: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- processed_at: timestamp (nullable = true)



customer_id,first_name,last_name,email,city,state,email_domain,full_name,processed_at
C00010,James,Myers,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,2026-10-17T11:36:31.649Z
C00045,Matthew,Lee,dkhan@hotmail.com,North Kara,OK,hotmail.com,Matthew Lee,2026-10-17T11:36:31.649Z
C00049,Mike,Harvey,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,2026-10-17T11:36:31.649Z
C00056,Tina,Cantu,cruzkristen@hotmail.com,Catherineberg,WA,hotmail.com,Tina Cantu,2026-10-17T11:36:31.649Z
C00062,Shelley,Holland,kimdennis@yahoo.com,Kaitlynburgh,MI,yahoo.com,Shelley Holland,2026-10-17T11:36:31.649Z
C00094,Hannah,Olson,rogersrandall@gonzales.org,Harrisonmouth,AK,gonzales.org,Hannah Olson,2026-10-17T11:36:31.649Z
C00104,Patrick,Meadows,gloria15@yahoo.com,New Danielle,VA,yahoo.com,Patrick Meadows,2026-10-17T11:36:31.649Z
C00105,April,Wright,christine52@kennedy.com,Crystalmouth,AL,kennedy.com,April Wright,2026-10-17T11:36:31.649Z
C00126,Amanda,King,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,2026-10-17T11:36:31.649Z
C00152,David,Salazar,tara25@guzman-nelson.net,Ashleyside,ID,guzman-nelson.net,David Salazar,2026-10-17T11:36:31.649Z


Rows: 2000


In [0]:
from pyspark.sql import functions as F

dim_customer = (
    spark.read.table("databricks_catalog.silver_schema.customers")
    .withColumn("customer_key", F.xxhash64("customer_id"))
    .select("customer_key", "customer_id", "first_name", "last_name", "full_name",
            "email", "email_domain", "city", "state")
    .withColumn("loaded_at", F.current_timestamp())
)

(dim_customer.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("databricks_catalog.gold_schema.dim_customer"))

print("✓ dim_customer written:", dim_customer.count(), "rows")

✓ dim_customer written: 2000 rows


In [0]:
%sql
SELECT * FROM databricks_catalog.gold_schema.dim_customer LIMIT 10;

SELECT COUNT(*) AS rows, COUNT(DISTINCT customer_id) AS unique_ids
FROM databricks_catalog.gold_schema.dim_customer;

rows,unique_ids
2000,2000
